In [0]:
config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
env_name = config.first()["env"].strip().lower()
lz_key = config.first()["lz_key"].strip().lower()

print(f"env_code: {lz_key}")  # This won't be redacted
print(f"env_name: {env_name}")  # This won't be redacted

KeyVault_name = f"ingest{lz_key}-meta002-{env_name}"
print(f"KeyVault_name: {KeyVault_name}") 


# Service principal credentials
client_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-ID")
client_secret = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id = dbutils.secrets.get(KeyVault_name, "SERVICE-PRINCIPLE-TENANT-ID")

# Storage account names
curated_storage = f"ingest{lz_key}curated{env_name}"
checkpoint_storage = f"ingest{lz_key}xcutting{env_name}"
raw_storage = f"ingest{lz_key}raw{env_name}"
landing_storage = f"ingest{lz_key}landing{env_name}"

# Spark config for curated storage (Delta table)
spark.conf.set(f"fs.azure.account.auth.type.{curated_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{curated_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{curated_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{curated_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{curated_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{checkpoint_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{checkpoint_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{checkpoint_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{checkpoint_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{checkpoint_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{raw_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{raw_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{raw_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{raw_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{raw_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# Spark config for checkpoint storage
spark.conf.set(f"fs.azure.account.auth.type.{landing_storage}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{landing_storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{landing_storage}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{landing_storage}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{landing_storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

# read_hive = False

# Setting variables for use in subsequent cells
raw_mnt = f"abfss://raw@ingest{lz_key}raw{env_name}.dfs.core.windows.net/ARIADM/ARM/JOH"
landing_mnt = f"abfss://landing@ingest{lz_key}landing{env_name}.dfs.core.windows.net/SQLServer/Sales/IRIS/dbo/"
bronze_mnt = f"abfss://bronze@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ARM/JOH"
silver_mnt = f"abfss://silver@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ARM/JOH"
gold_mnt = f"abfss://gold@ingest{lz_key}curated{env_name}.dfs.core.windows.net/ARIADM/ARM/JOH"
gold_outputs = "ARIADM/ARM/JOH"
hive_schema = "ariadm_arm_joh"
# key_vault = "ingest00-keyvault-sbox"

html_mnt = f"abfss://html-template@ingest{lz_key}landing{env_name}.dfs.core.windows.net/"

# Print all variables
variables = {
    # "read_hive": read_hive,
    "raw_mnt": raw_mnt,
    "landing_mnt": landing_mnt,
    "bronze_mnt": bronze_mnt,
    "silver_mnt": silver_mnt,
    "gold_mnt": gold_mnt,
    "html_mnt": html_mnt,
    "gold_outputs": gold_outputs,
    "hive_schema": hive_schema,
    "key_vault": KeyVault_name
}

display(variables)

try:
    env_value = dbutils.secrets.get(KeyVault_name, "Environment")
    env = "dev" if env_value == "development" else None
    print(f"Environment: {env}")
except:
    env = "unkown"

In [0]:
display(spark.sql("SHOW GRANT ON TABLE aria_stg01.ariadm_arm_joh.stg_create_joh_html_content"))

In [0]:
%sql

SELECT

  100.0 * SUM(
    CASE 
      WHEN regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           =
           regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS percent_normalised_match,

  100.0 * SUM(
    CASE 
      WHEN length(
             regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
           =
           length(
             regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS normalised_length_match_pct

FROM hive_metastore.ariadm_arm_joh.stg_create_joh_html_content t1
JOIN aria_stg01.ariadm_arm_joh.stg_create_joh_html_content t2

  ON t1.AdjudicatorId = t2.AdjudicatorId

In [0]:
%sql

SELECT

  100.0 * SUM(
    CASE 
      WHEN regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           =
           regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS percent_normalised_match,

  100.0 * SUM(
    CASE 
      WHEN length(
             regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
           =
           length(
             regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS normalised_length_match_pct

FROM hive_metastore.ariadm_arm_td.stg_create_td_iris_html_content t1
JOIN aria_stg01.ariadm_arm_td.stg_create_td_iris_html_content t2

  ON t1.CaseNo = t2.CaseNo

In [0]:
%sql

SELECT

  100.0 * SUM(
    CASE 
      WHEN regexp_replace(regexp_replace(t1.HTMLContent, '\\s+', ''), '&nbsp;', '')
           =
           regexp_replace(regexp_replace(t2.HTMLContent, '\\s+', ''), '&nbsp;', '')
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS percent_normalised_match,

  100.0 * SUM(
    CASE 
      WHEN length(
             regexp_replace(regexp_replace(t1.HTMLContent, '\\s+', ''), '&nbsp;', '')
           )
           =
           length(
             regexp_replace(regexp_replace(t2.HTMLContent, '\\s+', ''), '&nbsp;', '')
           )
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS normalised_length_match_pct

FROM hive_metastore.aria_bails.create_bails_html_content t1
JOIN aria_stg01.aria_bails.create_bails_html_content t2

  ON t1.CaseNo = t2.CaseNo

In [0]:
%sql

with t as (SELECT

  t1.CaseNo,
  length(regexp_replace(regexp_replace(t1.HTMLContent, '\\s+', ''), '&nbsp;', '')) AS meta_normalised_length,
  length(regexp_replace(regexp_replace(t2.HTMLContent, '\\s+', ''), '&nbsp;', '')) AS uc_normalised_length,

  case when length(regexp_replace(regexp_replace(t1.HTMLContent, '\\s+', ''), '&nbsp;', ''))
            =     
            length(regexp_replace(regexp_replace(t2.HTMLContent, '\\s+', ''), '&nbsp;', ''))
  then 1 else 0 end as normalised_length_match,

  case when regexp_replace(regexp_replace(t1.HTMLContent, '\\s+', ''), '&nbsp;', '')
            =
            regexp_replace(regexp_replace(t2.HTMLContent, '\\s+', ''), '&nbsp;', '')  
  then 1 else 0 end as normalised_match

FROM hive_metastore.aria_bails.create_bails_html_content t1
JOIN aria_stg01.aria_bails.create_bails_html_content t2
ON t1.CaseNo = t2.CaseNo)

select t.* from t where t.normalised_length_match = 0 

In [0]:

%sql
SELECT
  t1.CaseNo,

  length(
    regexp_replace(
      regexp_replace(t1.HTMLContent, '\\s+', ''),
      '&nbsp;',
      ''
    )
  ) AS meta_normalised_length,

  length(
    regexp_replace(
      regexp_replace(t2.HTMLContent, '\\s+', ''),
      '&nbsp;',
      ''
    )
  ) AS uc_normalised_length,

  regexp_replace(
    regexp_replace(t1.HTMLContent, '\\s+', ''),
    '&nbsp;',
    ''
  )
  =
  regexp_replace(
    regexp_replace(t2.HTMLContent, '\\s+', ''),
    '&nbsp;',
    ''
  ) AS normalised_match,
  t1.HTMLContent as metastore_html,
  t2.HTMLContent as uc_html

FROM hive_metastore.aria_bails.create_bails_html_content t1
JOIN aria_stg01.aria_bails.create_bails_html_content t2
  ON t1.CaseNo = t2.CaseNo

WHERE t1.CaseNo IN (
"HT/01056")



In [0]:
%sql

SELECT

  100.0 * SUM(
    CASE 
      WHEN regexp_replace(regexp_replace(t1.HTMLContent, '\\s+', ''), '&nbsp;', '')
           =
           regexp_replace(regexp_replace(t2.HTMLContent, '\\s+', ''), '&nbsp;', '')
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS percent_normalised_match,

  100.0 * SUM(
    CASE 
      WHEN length(
             regexp_replace(regexp_replace(t1.HTMLContent, '\\s+', ''), '&nbsp;', '')
           )
           =
           length(
             regexp_replace(regexp_replace(t2.HTMLContent, '\\s+', ''), '&nbsp;', '')
           )
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS normalised_length_match_pct

FROM hive_metastore.aria_s_bails.create_sbail_html_content t1
JOIN aria_stg01.aria_s_bails.create_sbail_html_content t2

  ON t1.CaseNo = t2.CaseNo

In [0]:
%sql

with t as (SELECT

  t1.CaseNo,
  length(regexp_replace(regexp_replace(t1.HTMLContent, '\\s+', ''), '&nbsp;', '')) AS meta_normalised_length,
  length(regexp_replace(regexp_replace(t2.HTMLContent, '\\s+', ''), '&nbsp;', '')) AS uc_normalised_length,

  case when length(regexp_replace(regexp_replace(t1.HTMLContent, '\\s+', ''), '&nbsp;', ''))
            =     
            length(regexp_replace(regexp_replace(t2.HTMLContent, '\\s+', ''), '&nbsp;', ''))
  then 1 else 0 end as normalised_length_match,

  case when regexp_replace(regexp_replace(t1.HTMLContent, '\\s+', ''), '&nbsp;', '')
            =
            regexp_replace(regexp_replace(t2.HTMLContent, '\\s+', ''), '&nbsp;', '')  
  then 1 else 0 end as normalised_match

FROM hive_metastore.aria_s_bails.create_sbail_html_content t1
JOIN aria_stg01.aria_s_bails.create_sbail_html_content t2
ON t1.CaseNo = t2.CaseNo)

select t.* from t where t.normalised_length_match = 0 


In [0]:
%sql

SELECT

  100.0 * SUM(
    CASE 
      WHEN regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           =
           regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS percent_normalised_match,

  100.0 * SUM(
    CASE 
      WHEN length(
             regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
           =
           length(
             regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS normalised_length_match_pct

FROM hive_metastore.ariadm_arm_fpa.stg_apl_create_html_content t1
JOIN aria_stg01.ariadm_arm_fpa.stg_apl_create_html_content t2

  ON t1.CaseNo = t2.CaseNo

In [0]:
%sql

SELECT

  100.0 * SUM(
    CASE 
      WHEN regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           =
           regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS percent_normalised_match,

  100.0 * SUM(
    CASE 
      WHEN length(
             regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
           =
           length(
             regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS normalised_length_match_pct

FROM hive_metastore.ariadm_arm_uta.stg_apl_create_html_content t1
JOIN aria_stg01.ariadm_arm_uta.stg_apl_create_html_content t2

  ON t1.CaseNo = t2.CaseNo

In [0]:
%sql

SELECT

  100.0 * SUM(
    CASE 
      WHEN regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           =
           regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS percent_normalised_match,

  100.0 * SUM(
    CASE 
      WHEN length(
             regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
           =
           length(
             regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')
           )
      THEN 1 ELSE 0 
    END
  ) / COUNT(*) AS normalised_length_match_pct

FROM hive_metastore.ariadm_arm_fta.stg_apl_create_html_content t1
JOIN aria_stg01.ariadm_arm_fta.stg_apl_create_html_content t2

  ON t1.CaseNo = t2.CaseNo

In [0]:
%sql

with t as (SELECT

  t1.CaseNo,
  length(regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')) AS meta_normalised_length,
  length(regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')) AS uc_normalised_length,

  case when length(regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', ''))
            =     
            length(regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', ''))
  then 1 else 0 end as normalised_length_match,

  case when regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
            =
            regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')  
  then 1 else 0 end as normalised_match

FROM hive_metastore.ariadm_arm_fta.stg_apl_create_html_content t1
JOIN aria_stg01.ariadm_arm_fta.stg_apl_create_html_content t2
ON t1.CaseNo = t2.CaseNo)

select t.* from t where t.normalised_length_match = 0 and t.CaseNo NOT IN ("AA/00476/2016",
"AA/02155/2014",
"AA/02920/2014",
"AA/03451/2013",
"AA/03908/2014",
"AA/11535/2011",
"AA/11904/2009",
"AA/11938/2009",
"DA/00070/2022",
"DA/00082/2021",
"DA/00082/2021",
"DA/00191/2020",
"DA/00231/2021",
"DA/00259/2021",
"EA/00995/2023",
"EA/00995/2023",
"EA/00995/2023",
"EA/00995/2023",
"EA/00996/2023",
"EA/00996/2023",
"EA/00996/2023",
"EA/00996/2023",
"EA/01495/2016",
"EA/01800/2021",
"EA/02107/2023",
"EA/03340/2021",
"EA/03395/2021",
"EA/03500/2022",
"EA/03627/2023",
"EA/03657/2023",
"EA/03678/2023",
"EA/03698/2016",
"EA/03771/2023",
"EA/03932/2023",
"EA/05058/2020",
"EA/06025/2021",
"EA/06563/2022",
"EA/06744/2022",
"EA/07478/2022",
"EA/07666/2021",
"EA/07953/2021",
"EA/08438/2022",
"EA/08440/2022",
"EA/09265/2022",
"EA/09369/2022",
"EA/09621/2022",
"EA/11098/2022",
"EA/12067/2022",
"EA/12644/2022",
"EA/12693/2022",
"EA/12694/2022",
"EA/12695/2022",
"EA/12696/2022",
"EA/12697/2022",
"EA/14463/2021",
"EA/15326/2021",
"EA/16206/2021",
"HU/00059/2024",
"HU/00174/2022",
"HU/00240/2020",
"HU/00499/2024",
"HU/00529/2025",
"HU/00535/2022",
"HU/00535/2022",
"HU/00548/2019",
"HU/00645/2024",
"HU/00710/2024",
"HU/00712/2022",
"HU/00780/2023",
"HU/00877/2024",
"HU/00884/2022",
"HU/00884/2022",
"HU/00893/2024",
"HU/01082/2022",
"HU/01275/2022",
"HU/01390/2022",
"HU/01523/2024",
"HU/01720/2020",
"HU/01804/2024",
"HU/01965/2022",
"HU/02109/2023",
"HU/02164/2023",
"HU/02179/2023",
"HU/02557/2021",
"HU/02693/2020",
"HU/04022/2020",
"HU/04745/2017",
"HU/04962/2016",
"HU/05693/2020",
"HU/05789/2019",
"HU/05789/2019",
"HU/05845/2020",
"HU/07201/2020",
"HU/07368/2020",
"HU/08250/2020",
"HU/08832/2019",
"HU/11302/2019",
"HU/11343/2015",
"HU/13946/2019",
"HU/13951/2019",
"HU/16916/2017",
"HU/18642/2019",
"HU/19742/2019",
"HU/20121/2019",
"HU/20312/2019",
"HU/20312/2019",
"HX/19526/2004",
"IA/01429/2014",
"IA/02028/2016",
"IA/02533/2020",
"IA/02582/2020",
"IA/03808/2010",
"IA/17832/2012",
"IA/27339/2014",
"IA/27339/2014",
"IA/27339/2014",
"IA/29096/2015",
"IA/29237/2015",
"IM/06108/2005",
"IM/14839/2005",
"IM/24963/2005",
"LE/04007/2024",
"LE/04320/2024",
"LE/04452/2024",
"LH/03658/2023",
"LP/00322/2022",
"LP/00935/2023",
"LP/06709/2024",
"LP/12645/2024",
"LR/00036/2022",
"OA/10886/2005",
"PA/00080/2024",
"PA/00315/2019",
"PA/00316/2019",
"PA/00348/2021",
"PA/00449/2020",
"PA/00457/2022",
"PA/00457/2022",
"PA/00531/2020",
"PA/00725/2023",
"PA/00860/2022",
"PA/00894/2024",
"PA/00916/2021",
"PA/00917/2023",
"PA/01022/2021",
"PA/01028/2023",
"PA/01028/2023",
"PA/01044/2022",
"PA/01083/2019",
"PA/01396/2024",
"PA/01443/2020",
"PA/01587/2021",
"PA/01591/2024",
"PA/01609/2023",
"PA/01647/2023",
"PA/01712/2023",
"PA/01918/2020",
"PA/02341/2019",
"PA/02430/2019",
"PA/02949/2018",
"PA/03334/2024",
"PA/03700/2020",
"PA/03754/2020",
"PA/04303/2024",
"PA/04303/2024",
"PA/04646/2024",
"PA/04728/2019",
"PA/05331/2019",
"PA/06719/2019",
"PA/07201/2019",
"PA/08197/2019",
"PA/08197/2019",
"PA/09297/2017",
"PA/10677/2019",
"PA/11360/2019",
"PA/11360/2019",
"PA/11516/2016",
"PA/12147/2018",
"PA/12480/2016",
"PA/12915/2017",
"PA/13444/2018",
"RP/00022/2023",
"RP/00028/2021")



In [0]:

%sql
SELECT
  t1.CaseNo,

  length(
    regexp_replace(
      regexp_replace(t1.HTML_Content, '\\s+', ''),
      '&nbsp;',
      ''
    )
  ) AS meta_normalised_length,

  length(
    regexp_replace(
      regexp_replace(t2.HTML_Content, '\\s+', ''),
      '&nbsp;',
      ''
    )
  ) AS uc_normalised_length,

  regexp_replace(
    regexp_replace(t1.HTML_Content, '\\s+', ''),
    '&nbsp;',
    ''
  )
  =
  regexp_replace(
    regexp_replace(t2.HTML_Content, '\\s+', ''),
    '&nbsp;',
    ''
  ) AS normalised_match,
  t1.HTML_Content as metastore_html,
  t2.HTML_Content as uc_html

FROM hive_metastore.ariadm_arm_fta.stg_apl_create_html_content t1

JOIN aria_stg01.ariadm_arm_fta.stg_apl_create_html_content t2
  ON t1.CaseNo = t2.CaseNo

WHERE t1.CaseNo IN (
  -- "LE/01109/2023",
"PA/04406/2024"
-- "LE/01110/2023",
-- "IA/09595/2021",
-- "IA/09595/2021",
-- "LE/01108/2023",
-- "HU/07301/2020",
-- "IA/09536/2022",
-- "RP/00011/2024"
)



In [0]:
%sql

with t as (SELECT

  t1.CaseNo,
  length(regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')) AS meta_normalised_length,
  length(regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')) AS uc_normalised_length,

  case when length(regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', ''))
            =     
            length(regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', ''))
  then 1 else 0 end as normalised_length_match,

  case when regexp_replace(regexp_replace(t1.HTML_Content, '\\s+', ''), '&nbsp;', '')
            =
            regexp_replace(regexp_replace(t2.HTML_Content, '\\s+', ''), '&nbsp;', '')  
  then 1 else 0 end as normalised_match

  -- t1.HTML_Content AS metastore_html,
  -- t2.HTML_Content AS uc_html

FROM hive_metastore.ariadm_arm_uta.stg_apl_create_html_content t1
JOIN aria_stg01.ariadm_arm_uta.stg_apl_create_html_content t2
ON t1.CaseNo = t2.CaseNo)

select t.* from t where t.normalised_length_match = 0



In [0]:

%sql
SELECT
  t1.CaseNo,

  length(
    regexp_replace(
      regexp_replace(t1.HTML_Content, '\\s+', ''),
      '&nbsp;',
      ''
    )
  ) AS meta_normalised_length,

  length(
    regexp_replace(
      regexp_replace(t2.HTML_Content, '\\s+', ''),
      '&nbsp;',
      ''
    )
  ) AS uc_normalised_length,

  regexp_replace(
    regexp_replace(t1.HTML_Content, '\\s+', ''),
    '&nbsp;',
    ''
  )
  =
  regexp_replace(
    regexp_replace(t2.HTML_Content, '\\s+', ''),
    '&nbsp;',
    ''
  ) AS normalised_match,
  t1.HTML_Content as metastore_html,
  t2.HTML_Content as uc_html

FROM hive_metastore.ariadm_arm_uta.stg_apl_create_html_content t1

JOIN aria_stg01.ariadm_arm_uta.stg_apl_create_html_content t2
  ON t1.CaseNo = t2.CaseNo

WHERE t1.CaseNo IN (
"DA/00110/2019")

